# Fire Hazard: BBox vs Segmentation Experiment

Controlled Colab workflow for comparing YOLO26 bounding boxes and YOLO26 segmentation masks with a shared YOLO26-Depth map per image. Run `00_environment_check.ipynb` first and only continue when it reports `READY FOR FULL EXPERIMENT: YES`.


## 1. Runtime Setup
Use a GPU runtime in Colab. This notebook assumes the repository has been cloned to `/content/fire-hazard-spatial`.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json

REPO_ROOT = Path('/content/fire-hazard-spatial')
if not REPO_ROOT.exists():
    raise FileNotFoundError('Repository not found at /content/fire-hazard-spatial. Clone the repo first.')
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
print('Repository:', REPO_ROOT)


In [ ]:
!pip -q install -r requirements.txt


## 2. Paths and Experiment Settings
These are the paths that passed your environment check. Change only if you move folders in Google Drive.


In [ ]:
from pathlib import Path

SEG_SOURCE_DATASET_ROOT = Path('/content/drive/MyDrive/fire_hazard_dataset/Fire Hazard YOLO26')
SEG_CLEAN_DATASET_ROOT = Path('/content/drive/MyDrive/fire_hazard_dataset/Fire Hazard YOLO26 Segmentation Clean')
DET_DATASET_ROOT = Path('/content/drive/MyDrive/fire_hazard_dataset/Fire Hazard YOLO26 Detection')
OUTPUT_ROOT = Path('/content/drive/MyDrive/fire_hazard_experiment_outputs')

DET_DATA_YAML = DET_DATASET_ROOT / 'data.yaml'
SEG_DATA_YAML = SEG_CLEAN_DATASET_ROOT / 'data.yaml'

RUN_SMOKE_TEST = True
RUN_TRAINING = True
RUN_EVALUATION = True
RUN_DEPTH_AND_FUSION = True

YOLO26_DET_MODEL = 'yolo26n.pt'
YOLO26_SEG_MODEL = 'yolo26n-seg.pt'
YOLO26_DEPTH_MODEL = 'yolo26n-depth.pt'

IMGSZ = 640
EPOCHS = 75
BATCH = 8
SEED = 42
CONF = 0.25
IOU_MATCH_THRESHOLD = 0.30

print('Detection dataset:', DET_DATASET_ROOT)
print('Segmentation dataset:', SEG_CLEAN_DATASET_ROOT)
print('Output root:', OUTPUT_ROOT)


## 3. Validate Datasets
This checks both converted detection labels and clean segmentation polygon labels before training.


In [ ]:
import subprocess, sys

checks = [
    ('detection', DET_DATASET_ROOT, DET_DATA_YAML),
    ('segmentation', SEG_CLEAN_DATASET_ROOT, SEG_DATA_YAML),
]
for mode, root, data_yaml in checks:
    print('\nVALIDATING', mode.upper())
    cmd = [
        sys.executable, 'scripts/check_dataset.py',
        '--dataset-root', str(root),
        '--data-yaml', str(data_yaml),
        '--task-mode', mode,
    ]
    subprocess.run(cmd, check=True)


## 4. Smoke Test
The smoke test runs one image through detection, segmentation, and depth. Do not continue to full training if this fails.


In [ ]:
from experiment_core import ensure_dirs, smoke

ensure_dirs(OUTPUT_ROOT)
if RUN_SMOKE_TEST:
    smoke(
        YOLO26_DET_MODEL, YOLO26_SEG_MODEL, YOLO26_DEPTH_MODEL,
        DET_DATASET_ROOT, SEG_CLEAN_DATASET_ROOT, OUTPUT_ROOT, IMGSZ, CONF
    )
    print('Smoke test passed. OK to continue.')
else:
    print('Smoke test skipped by RUN_SMOKE_TEST = False')


## 5. Train Detection and Segmentation
Both models use the same image size, seed, batch size, and epoch count.


In [ ]:
from experiment_core import yolo_train

if RUN_TRAINING:
    yolo_train(YOLO26_DET_MODEL, DET_DATA_YAML, OUTPUT_ROOT, 'detect', IMGSZ, EPOCHS, BATCH, SEED)
    yolo_train(YOLO26_SEG_MODEL, SEG_DATA_YAML, OUTPUT_ROOT, 'segment', IMGSZ, EPOCHS, BATCH, SEED)
else:
    print('Training skipped.')


## 6. Evaluate Models


In [ ]:
from experiment_core import yolo_eval

DET_BEST = OUTPUT_ROOT / 'detect' / 'train' / 'weights' / 'best.pt'
SEG_BEST = OUTPUT_ROOT / 'segment' / 'train' / 'weights' / 'best.pt'

if RUN_EVALUATION:
    yolo_eval(DET_BEST, DET_DATA_YAML, OUTPUT_ROOT / 'tables' / 'detection_metrics.csv', 'detect', IMGSZ, CONF)
    yolo_eval(SEG_BEST, SEG_DATA_YAML, OUTPUT_ROOT / 'tables' / 'segmentation_metrics.csv', 'segment', IMGSZ, CONF)
else:
    print('Evaluation skipped.')


## 7. Depth and Spatial Evidence Tables
Depth is treated as relative unless you verify that the selected YOLO26-Depth model returns metric depth.


In [ ]:
from experiment_core import run_depth, pred_table, bbox_depth, mask_depth, match_instances, compare_depth, pairs, summary
import pandas as pd

if RUN_DEPTH_AND_FUSION:
    run_depth(YOLO26_DEPTH_MODEL, DET_DATASET_ROOT, OUTPUT_ROOT, IMGSZ)
    det_pred = pred_table(DET_BEST, DET_DATASET_ROOT, DET_DATA_YAML, OUTPUT_ROOT, 'detect', IMGSZ, CONF)
    seg_pred = pred_table(SEG_BEST, SEG_CLEAN_DATASET_ROOT, SEG_DATA_YAML, OUTPUT_ROOT, 'segment', IMGSZ, CONF)
    bbox_csv = bbox_depth(det_pred, OUTPUT_ROOT)
    mask_csv = mask_depth(seg_pred, OUTPUT_ROOT)
    matches_csv = match_instances(bbox_csv, mask_csv, OUTPUT_ROOT, IOU_MATCH_THRESHOLD)
    object_csv = compare_depth(matches_csv, bbox_csv, mask_csv, OUTPUT_ROOT)
    pair_csv = pairs(object_csv, OUTPUT_ROOT)
    try:
        summary(object_csv, pair_csv, OUTPUT_ROOT)
    except pd.errors.EmptyDataError:
        obj_count = len(pd.read_csv(object_csv)) if object_csv.exists() and object_csv.stat().st_size > 0 else 0
        pair_count = 0
        summary_rows = [
            {'Metric': 'matched instances', 'BBox': obj_count, 'Mask': obj_count, 'Difference': 0},
            {'Metric': 'object pairs', 'BBox': pair_count, 'Mask': pair_count, 'Difference': 0},
            {'Metric': 'note', 'BBox': 'Not enough matched instances in the same image to form object pairs. This is valid for a very small feasibility run.', 'Mask': '', 'Difference': ''},
        ]
        (OUTPUT_ROOT / 'tables').mkdir(parents=True, exist_ok=True)
        pd.DataFrame(summary_rows).to_csv(OUTPUT_ROOT / 'tables' / 'bbox_vs_mask_summary.csv', index=False)
        # Replace the empty pair files with header-only CSVs so later display cells do not crash.
        pair_columns = ['image_id', 'object_A_id', 'object_A_class', 'object_B_id', 'object_B_class', 'bbox_depth_difference', 'mask_depth_difference', 'ground_truth_relation', 'annotator_notes']
        pd.DataFrame(columns=pair_columns).to_csv(OUTPUT_ROOT / 'tables' / 'object_pair_comparison.csv', index=False)
        pd.DataFrame(columns=pair_columns).to_csv(OUTPUT_ROOT / 'tables' / 'spatial_relation_annotation_template.csv', index=False)
        print(f'Only {obj_count} matched instance(s); wrote header-only pair tables and a valid summary.')
    print('Experiment tables written to:', OUTPUT_ROOT / 'tables')
else:
    print('Depth/fusion skipped.')


## 8. Inspect Key Outputs


In [ ]:
import pandas as pd

for name in [
    'detection_metrics.csv',
    'segmentation_metrics.csv',
    'bbox_vs_mask_summary.csv',
    'object_depth_comparison.csv',
    'instance_matches.csv',
    'object_pair_comparison.csv',
    'spatial_relation_annotation_template.csv',
]:
    path = OUTPUT_ROOT / 'tables' / name
    print('\n', path)
    if path.exists():
        try:
            display(pd.read_csv(path).head())
        except pd.errors.EmptyDataError:
            print('File exists but has no rows/columns. This usually means no comparable pairs were produced.')
    else:
        print('Not created yet.')


## 9. Notes for Thesis Reporting
Use conservative wording: this run checks feasibility and evidence quality on a small dataset, not final model superiority. Spatial evidence from monocular depth is relative unless metric depth is verified.
